# SHQ Analysis Program 1 - Analyze MVIC files from dynamometer
Updated 04/17/2026

----- 
## Cell 1 - Imports
Import the necessary packages to run the program. 

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tkinter import Tk
import re
from tkinter.filedialog import askdirectory, asksaveasfilename
from scipy.signal import butter, filtfilt
import sys
import qdarktheme
import matplotlib
matplotlib.use('QtAgg')
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator
from PyQt6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout,
                              QHBoxLayout, QLabel, QPushButton, QSlider, QProgressBar, QSizePolicy)
from PyQt6.QtCore import Qt
from PyQt6.QtGui import QFont

print(f'{datetime.now()} - Imports OK')

# Also defining custom plot theme in this code block
custom_theme = {"axes.spines.right": False, 
                "axes.spines.top": False,
                "axes.titlelocation": "left", 
                "axes.titley": 1,
                "font.weight":"bold", 
                "axes.titlesize": "x-large", 
                "axes.labelsize": "x-large",
                "axes.titleweight": "bold", 
                "axes.labelweight": 'bold'}

plt.rcParams.update({
    **custom_theme,
    'figure.facecolor': 'white',  # transparent background for plots
    'axes.labelcolor': 'black',   # White axes labels
    'axes.edgecolor': 'black',    # White axes edge color
    'xtick.color': 'black',       # White x-axis tick labels
    'ytick.color': 'black',       # white y-axis tick labels
    "axes.titlecolor": "black"    # white title label
})

2026-04-20 20:24:09.755991 - Imports OK


------ 
## Cell 2 - Define Functions
Define the custom functions to use. 

The first function - `butter_lowpass_torque`- is a basic 4th order zero-lag Butterworth filter to filter the torque data. 

The second function - `mvic_window` - finds the windowed MVIC (the highest 0.25 s average across the entire file). We do this to avoid any "peaks" in a MVIC curve being erroneously called the max value. 

The last function - `pick_directory` - is the normal clean function I use to choose the folder in which the participant's data is stored. 

In [ ]:
# First, define the filter function 
# Constants 
sf = 2000
fc = 10 # aplying a cutoff frequency of 10 Hz first. Will adjust as needed.
nyf = 0.5 * sf
order = 2 ## zero lag filter so needs to be 2 to make a 4th order filter 
cn = fc/nyf

def butter_lowpass_torque(torque, cutoff, fs=sf, order=2):
    """
    Zero-phase Butterworth low-pass filter
    data = array you want to filter
    fs = sample frequency
    order = butterworth order 2 because filtered twice
    cutoff = lowpass cutoff frequency
    """
    b, a = butter(order, cutoff / (fs / 2.0), btype='low')
    return filtfilt(b, a, torque, axis=0)

# Find the max 250 ms (0.25s) MVIC with a sliding window function
def mvic_window(torque, epoch_dur=0.25, sf=sf):
    mvic_array = []
    epoch_samples = int(epoch_dur * sf)
    for start_idx in range(len(torque) - epoch_samples):
        end_idx = start_idx + epoch_samples
        epoch_avg = np.mean(torque[start_idx:end_idx])
        mvic_array.append(epoch_avg)

    max_windowed = max(mvic_array)

    return max_windowed

# Define the folder finding function. 
def pick_directory(prompt = "Select the Participant's Data Folder (e.g., SHQ002)"):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    folder = askdirectory(title=prompt)
    root.destroy()
    return folder

# now pick the folder to save the analyzed data to
def pick_save_path(default_name):
    '''Open a save as dialog'''
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    path = asksaveasfilename(
        title = "Choose where to save the MVIC results",
        initialfile = default_name,
        defaultextension = '.csv',
        filetypes = [('Excel files')])
    root.destroy()

    return path

# Next function - parsing trial name
def parse_trial_name(trial_name):
    """
    Parse a filename like 'LEFT_KE1' or 'RIGHT_KF3' into
    its component parts: leg, action, rep.
    Returns a dict with keys: Leg, Action, Rep
    """
    match = re.match(r'(LEFT|RIGHT)_(KE|KF)(\d+)', trial_name, re.IGNORECASE)
    if match:
        return {
            'Leg':    match.group(1).capitalize(),
            'Action': match.group(2).upper(),
            'Rep':    int(match.group(3))
        }
    return {'Leg': None, 'Action': None, 'Rep': None}


------ 
## Cell 3 - Select the participant's folder and pull on the MVIC files
This will only select the MVIC files stored in the "Full Data" subfolder. 

In [77]:
participant_folder = pick_directory()

participant_id = os.path.basename(participant_folder)

fulldata_folder = os.path.join(participant_folder, "Full Data")

full_files = os.listdir(fulldata_folder)

mvic_files = [f for f in full_files if "KE" in f or "KF" in f]

print(f'Folder is: {participant_folder}. With the following MVIC files...')
for f in mvic_files:
    print(f'      - {f[:-4]}')

# Now, pull all the MVIC files into a dictionary
mvic_dict = {}

# Read each trial in the folder
for mvic_trial in mvic_files:
    full_name = os.path.join(fulldata_folder, mvic_trial)
    full_dat = pd.read_csv(full_name)
    torque = full_dat.iloc[:, 2].values # pull only the torque column
    torque = torque[~np.isnan(torque)]
    cleaned_name = mvic_trial[:-4]
    mvic_dict[cleaned_name] = torque

Folder is: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection/SHQ022. With the following MVIC files...
      - LEFT_KE1
      - LEFT_KE2
      - LEFT_KE3
      - LEFT_KF1
      - LEFT_KF2
      - LEFT_KF3
      - RIGHT_KE1
      - RIGHT_KE2
      - RIGHT_KE3
      - RIGHT_KF1
      - RIGHT_KF2
      - RIGHT_KF3


------
## Cell 4 - Interactive baseline correction and MVIC extraction

This cell steps through **every trial one at a time**. For each trial:

1. The signal is plotted with two vertical cursor lines — a **blue START** cursor and an **orange END** cursor.
2. Use the **Start** and **End sliders** to position the cursors over the quiet baseline period (before the contraction ramps up).
3. The shaded region and baseline mean update live as you drag.
4. Select **Reset sliders** from the dropdown to return cursors to their default positions.

> **Tip:** The default window is placed at the very start of the file where the participant is typically quiet before the cue. Widen or shift as needed based on the recording.

In [78]:
# ── Match your existing plot theme ────────────────────────────────────────────
plt.rcParams.update({
    **custom_theme,
    'figure.facecolor': 'none',
    'axes.facecolor':   'none',
    'axes.labelcolor':  '#ffffff',
    'axes.edgecolor':   '#ffffff',
    'xtick.color':      '#ffffff',
    'ytick.color':      '#ffffff',
    'axes.titlecolor':  'white',
    'legend.labelcolor':'white',
    'legend.facecolor': 'none',
    'legend.edgecolor': 'none',
})

results     = []
trial_names = list(mvic_dict.keys())

BTN_STYLE_PRIMARY = """
    QPushButton {
        background-color: #4caf50;
        color: white;
        font-weight: bold;
        font-size: 13px;
        padding: 8px 20px;
        border-radius: 4px;
        border: none;
    }
    QPushButton:hover    { background-color: #43a047; }
    QPushButton:pressed  { background-color: #388e3c; }
    QPushButton:disabled { background-color: #555; color: #888; }
"""
BTN_STYLE_APPLY = """
    QPushButton {
        background-color: #2196f3;
        color: white;
        font-weight: bold;
        font-size: 13px;
        padding: 8px 20px;
        border-radius: 4px;
        border: none;
    }
    QPushButton:hover    { background-color: #1e88e5; }
    QPushButton:pressed  { background-color: #1565c0; }
    QPushButton:disabled { background-color: #555; color: #888; }
"""
BTN_STYLE_SECONDARY = """
    QPushButton {
        background-color: #e67e22;
        color: white;
        font-weight: bold;
        font-size: 13px;
        padding: 8px 20px;
        border-radius: 4px;
        border: none;
    }
    QPushButton:hover   { background-color: #d35400; }
    QPushButton:pressed { background-color: #ba4a00; }
"""
LABEL_STYLE  = "color: white; font-size: 13px;"
STATUS_STYLE = "color: #aaaaaa; font-size: 12px;"
MVIC_STYLE   = "color: #4caf50; font-size: 13px; font-weight: bold;"
PROG_STYLE   = """
    QProgressBar {
        border: 1px solid #555;
        border-radius: 4px;
        text-align: center;
        color: white;
        background: #2b2b2b;
    }
    QProgressBar::chunk {
        background-color: #4caf50;
        border-radius: 3px;
    }
"""

class MVICWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.i            = 0
        self.trial_name   = None
        self.torque_raw   = None
        self.t            = None
        self.duration     = None
        self.current_mvic = None      # set after Apply Offset
        self.current_base = None

        self.setWindowTitle('SHQ — MVIC Analysis')
        self.setMinimumSize(1200, 800)
        self.showMaximized()

        # ── Central widget ────────────────────────────────────────────────────
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout(central)
        main_layout.setSpacing(8)
        main_layout.setContentsMargins(20, 14, 20, 14)

        # ── Trial label ───────────────────────────────────────────────────────
        self.trial_label = QLabel('')
        self.trial_label.setFont(QFont('Arial', 14, QFont.Weight.Bold))
        self.trial_label.setStyleSheet(LABEL_STYLE)
        main_layout.addWidget(self.trial_label)

        # ── Canvas 1 — raw signal with baseline cursors ───────────────────────
        self.fig1   = Figure(figsize=(12, 3))
        self.ax1    = self.fig1.add_subplot(111)
        self.canvas1 = FigureCanvas(self.fig1)
        self.canvas1.setSizePolicy(QSizePolicy.Policy.Expanding,
                            QSizePolicy.Policy.Expanding)
        self.fig1.tight_layout(pad=1.5)
        main_layout.addWidget(self.canvas1, stretch=4)

        # ── Sliders ───────────────────────────────────────────────────────────
        for attr, lbl_attr, lbl_text in [
            ('sld_start', 'lbl_start', 'Baseline Start:'),
            ('sld_end',   'lbl_end',   'Baseline End:  '),
        ]:
            row = QHBoxLayout()
            lbl = QLabel(lbl_text)
            lbl.setFont(QFont('Arial', 11))
            lbl.setStyleSheet(LABEL_STYLE)
            lbl.setFixedWidth(140)
            setattr(self, lbl_attr, lbl)

            sld = QSlider(Qt.Orientation.Horizontal)
            sld.setMinimum(0)
            sld.setMaximum(10000)
            sld.valueChanged.connect(self.on_slider)
            setattr(self, attr, sld)

            val_lbl = QLabel('0.000 s')
            val_lbl.setFont(QFont('Arial', 11))
            val_lbl.setStyleSheet(LABEL_STYLE)
            val_lbl.setFixedWidth(75)
            setattr(self, lbl_attr + '_val', val_lbl)

            row.addWidget(lbl)
            row.addWidget(sld)
            row.addWidget(val_lbl)
            main_layout.addLayout(row)

        # ── Status label ──────────────────────────────────────────────────────
        self.status_label = QLabel('')
        self.status_label.setStyleSheet(STATUS_STYLE)
        self.status_label.setFont(QFont('Arial', 11))
        main_layout.addWidget(self.status_label)

        # ── Buttons ───────────────────────────────────────────────────────────
        btn_row = QHBoxLayout()
        btn_row.setSpacing(12)

        self.btn_reset = QPushButton('Reset Sliders')
        self.btn_reset.setStyleSheet(BTN_STYLE_SECONDARY)
        self.btn_reset.clicked.connect(self.on_reset)

        self.btn_apply = QPushButton('Apply Offset')
        self.btn_apply.setStyleSheet(BTN_STYLE_APPLY)
        self.btn_apply.clicked.connect(self.on_apply)

        self.btn_next = QPushButton('Next Trial')
        self.btn_next.setStyleSheet(BTN_STYLE_PRIMARY)
        self.btn_next.setEnabled(False)      # disabled until offset applied
        self.btn_next.clicked.connect(self.on_next)

        btn_row.addWidget(self.btn_reset)
        btn_row.addWidget(self.btn_apply)
        btn_row.addWidget(self.btn_next)
        btn_row.addStretch()
        main_layout.addLayout(btn_row)

        # ── Canvas 2 — filtered corrected signal ──────────────────────────────
        self.fig2    = Figure(figsize=(12, 3))
        self.ax2     = self.fig2.add_subplot(111)
        self.canvas2 = FigureCanvas(self.fig2)
        self.canvas2.setSizePolicy(QSizePolicy.Policy.Expanding,
                            QSizePolicy.Policy.Expanding)
        self.fig2.tight_layout(pad=1.5)
        main_layout.addWidget(self.canvas2, stretch=4)

        # ── MVIC result label ─────────────────────────────────────────────────
        self.mvic_label = QLabel('')
        self.mvic_label.setStyleSheet(MVIC_STYLE)
        self.mvic_label.setFont(QFont('Arial', 11, QFont.Weight.Bold))
        main_layout.addWidget(self.mvic_label)

        # ── Progress bar ──────────────────────────────────────────────────────
        self.progress = QProgressBar()
        self.progress.setMinimum(0)
        self.progress.setMaximum(len(trial_names))
        self.progress.setValue(0)
        self.progress.setFixedHeight(22)
        self.progress.setStyleSheet(PROG_STYLE)
        self.progress.setFormat(f'%v / {len(trial_names)} trials')
        main_layout.addWidget(self.progress)

        # ── Load first trial ──────────────────────────────────────────────────
        self.load_trial(0)

    def resizeEvent(self, event):
        super().resizeEvent(event)
        try:
            if self.torque_raw is not None:
                x0 = self.sld_start.value() / 1000
                x1 = self.sld_end.value()   / 1000
            if x0 < x1:
                self.redraw_raw(x0, x1)
        except Exception:
            pass 
    # ── Load trial ────────────────────────────────────────────────────────────
    def load_trial(self, i):
        self.trial_name   = trial_names[i]
        self.torque_raw   = mvic_dict[self.trial_name]
        self.t            = np.arange(len(self.torque_raw)) / sf
        self.duration     = self.t[-1]
        self.current_mvic = None
        self.current_base = None

        max_ms      = int(self.duration * 1000)
        default_end = int(min(500, max_ms * 0.1))

        self.sld_start.valueChanged.disconnect()
        self.sld_end.valueChanged.disconnect()
        self.sld_start.setMaximum(max_ms)
        self.sld_end.setMaximum(max_ms)
        self.sld_start.setValue(0)
        self.sld_end.setValue(default_end)
        self.sld_start.valueChanged.connect(self.on_slider)
        self.sld_end.valueChanged.connect(self.on_slider)

        self.trial_label.setText(
            f'Trial {i+1} / {len(trial_names)}  —  {self.trial_name}'
        )
        self.mvic_label.setText('')
        self.lbl_start_val.setText('0.000 s')
        self.lbl_end_val.setText(f'{default_end/1000:.3f} s')
        self.btn_next.setEnabled(False)

        # Clear canvas 2 for new trial
        self.ax2.clear()
        self.canvas2.draw()

        self.redraw_raw(0.0, default_end / 1000)

    # ── Redraw raw signal (canvas 1) ──────────────────────────────────────────
    def redraw_raw(self, x0, x1):
        t   = self.t
        tor = self.torque_raw
        i0  = np.searchsorted(t, x0)
        i1  = np.searchsorted(t, x1)
        seg = tor[i0:i1]
        baseline_val = float(seg.mean()) if len(seg) > 0 else 0.0

        self.ax1.clear()
        self.ax1.plot(t, tor, color='#5b9bd5', lw=1.2)
        self.ax1.axhline(0, color='#666666', lw=0.5, ls='--')
        self.ax1.axvline(x0, color='#5b9bd5', lw=2, ls='--')
        self.ax1.axvline(x1, color='#e67e22', lw=2, ls='--')
        self.ax1.axvspan(x0, x1, alpha=0.18, facecolor='#5b9bd5')
        self.ax1.axhline(baseline_val, color='#e74c3c', lw=1.2, ls=':')
        self.ax1.set_xlabel('Time (s)')
        self.ax1.set_ylabel('Torque (Nm)')
        self.ax1.set_title(f'Raw Signal — {self.trial_name}')
        self.ax1.spines['top'].set_visible(False)
        self.ax1.spines['right'].set_visible(False)
        self.ax1.xaxis.set_major_locator(MaxNLocator(integer=False, prune='both'))
        self.ax1.yaxis.set_major_locator(MaxNLocator(nbins='auto', prune='both'))
        self.ax1.legend(fontsize=9, loc='upper right', frameon=False)
        self.fig1.tight_layout()
        self.canvas1.draw()

        self.status_label.setText(
            f'Window: {x0:.3f} – {x1:.3f} s  '
            f'({(x1-x0)*1000:.0f} ms)  |  '
            f'Baseline mean: {baseline_val:.4f} Nm'
        )

    # ── Draw corrected/filtered signal (canvas 2) ─────────────────────────────
    def draw_corrected(self, torque_corr, torque_filt, mvic_val, baseline_val):
        t           = self.t
        mvic_array  = []
        epoch_samps = int(0.25 * sf)
        for s in range(len(torque_filt) - epoch_samps):
            mvic_array.append(np.mean(torque_filt[s:s + epoch_samps]))
        mvic_idx        = int(np.argmax(mvic_array))
        mvic_start_time = t[mvic_idx]
        mvic_end_time   = mvic_start_time + 0.25
        annotation_x    = (mvic_start_time + mvic_end_time) / 2
        annotation_y    = float(torque_filt.max()) * 0.5

        self.ax2.clear()
        self.ax2.plot(t, torque_corr, color='#5b9bd5', lw=0.5,
                      ls='--', label='Offset corrected')
        self.ax2.plot(t, torque_filt, color='white', lw=1.8,
                      label='Filtered')
        self.ax2.axhline(0, color='#666666', lw=0.5, ls='--')
        self.ax2.axvspan(mvic_start_time, mvic_end_time,
                         color='#e67e22', alpha=0.3)
        self.ax2.annotate(
            f'MVIC\n{mvic_val:.2f} Nm',
            xy=(annotation_x, annotation_y),
            xytext=(annotation_x, annotation_y + 0.04 * float(torque_filt.max())),
            ha='center', fontsize=9, color='#e67e22',
            fontweight='bold'
        )
        self.ax2.set_xlabel('Time (s)')
        self.ax2.set_ylabel('Torque (Nm)')
        self.ax2.set_title(
            f'Offset Corrected & Filtered — {self.trial_name}  '
            f'(Baseline = {baseline_val:.4f} Nm)'
        )
        self.ax2.spines['top'].set_visible(False)
        self.ax2.spines['right'].set_visible(False)
        self.ax2.xaxis.set_major_locator(MaxNLocator(integer=False, prune='both'))
        self.ax2.yaxis.set_major_locator(MaxNLocator(nbins='auto', prune='both'))
        self.ax2.legend(fontsize=9, loc='upper right', frameon=False)
        self.fig2.tight_layout()
        self.canvas2.draw()

    # ── Slider changed ────────────────────────────────────────────────────────
    def on_slider(self):
        x0 = self.sld_start.value() / 1000
        x1 = self.sld_end.value()   / 1000
        self.lbl_start_val.setText(f'{x0:.3f} s')
        self.lbl_end_val.setText(  f'{x1:.3f} s')
        if x0 >= x1:
            self.status_label.setText('ERROR: Start must be before End.')
            return
        # Moving sliders after applying disables Next until re-applied
        self.btn_next.setEnabled(False)
        self.mvic_label.setText('')
        self.redraw_raw(x0, x1)

    # ── Reset ─────────────────────────────────────────────────────────────────
    def on_reset(self):
        max_ms      = int(self.duration * 1000)
        default_end = int(min(500, max_ms * 0.1))
        self.sld_start.setValue(0)
        self.sld_end.setValue(default_end)

    # ── Apply Offset ──────────────────────────────────────────────────────────
    def on_apply(self):
        x0 = self.sld_start.value() / 1000
        x1 = self.sld_end.value()   / 1000

        if x0 >= x1:
            self.status_label.setText('ERROR: Fix slider positions first.')
            return

        t   = self.t
        tor = self.torque_raw
        i0  = np.searchsorted(t, x0)
        i1  = np.searchsorted(t, x1)

        baseline_val      = tor[i0:i1].mean()
        torque_corr       = tor - baseline_val
        torque_filt       = butter_lowpass_torque(torque_corr, cutoff=fc)
        mvic_val          = mvic_window(torque_filt)
        self.current_mvic = mvic_val
        self.current_base = baseline_val
        self.current_x0   = x0
        self.current_x1   = x1

        self.draw_corrected(torque_corr, torque_filt, mvic_val, baseline_val)

        self.mvic_label.setText(
            f'MVIC = {mvic_val:.3f} N·m  |  '
            f'Baseline = {baseline_val:.4f} N·m  —  '
            f'Happy with this? Click Next Trial to save.'
        )
        self.btn_next.setEnabled(True)

    # ── Next Trial ────────────────────────────────────────────────────────────
    def on_next(self):
        parsed = parse_trial_name(self.trial_name)
        results.append({
            'ID':               participant_id,
            'Leg':              parsed['Leg'],
            'Action':           parsed['Action'],
            'Rep':              parsed['Rep'],
            'MVIC_Nm':          round(self.current_mvic, 4),
        })

        self.i += 1
        self.progress.setValue(self.i)

        if self.i < len(trial_names):
            self.load_trial(self.i)
        else:
            self.trial_label.setText('✓  All trials complete — close this window.')
            self.btn_apply.setEnabled(False)
            self.btn_next.setEnabled(False)

# ── Launch ─────────────────────────────────────────────────────────────────────
app = QApplication.instance() or QApplication(sys.argv)
app.setStyleSheet(qdarktheme.load_stylesheet())
win = MVICWindow()
win.show()
app.exec()

print(f'\nDone — {len(results)} trials collected.')
print('Proceed to Cell 5 to save.')

<positron-console-cell-78>:268: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.



Done — 12 trials collected.
Proceed to Cell 5 to save.


------
## Cell 5 - Save results to Excel

Run this cell only after **all trials have been confirmed** in Cell 4.

A Save As dialog will open. The filename is pre-filled with today's date. If a results file for today already exists at the chosen path (i.e., you processed another participant today), the new rows will be **appended** so data accumulates across participants over the collection period.

In [79]:
if not results:
    print("No results to save — complete Cell 4 first.")
else:
    today_str    = datetime.now().strftime('%Y-%m-%d')
    default_name = f"SHQ_MVIC_results_{today_str}.csv"
    save_path    = pick_save_path(default_name)

    if not save_path:
        print("Save cancelled.")
    else:
        col_order = ['ID', 'Leg', 'Action', 'Rep', 'MVIC_Nm']
        new_df = pd.DataFrame(results)[col_order]

        if os.path.exists(save_path):
            existing_df = pd.read_csv(save_path)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            # Keep last entry for any duplicate ID/Leg/Action/Rep combination 
            combined_df = combined_df.drop_duplicates(
                subset=['ID', 'Leg', 'Action', 'Rep'],
                keep='last')

            print(f"Existing file found — appending {len(new_df)} rows "
                     f"({len(existing_df)} already present, duplicates removed).")
        else:
            combined_df = new_df
            print(f"Creating new file with {len(new_df)} rows.")

        combined_df.to_csv(save_path, index=False)
        print(f"Data saved to: {save_path}")
        display(new_df)

Existing file found — appending 12 rows (251 already present, duplicates removed).
Data saved to: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection/SHQ_MVIC_results_2026-04-20.csv


,ID,Leg,Action,Rep,MVIC_Nm
0,SHQ022,Left,KE,1,274.5502
1,SHQ022,Left,KE,2,237.8319
2,SHQ022,Left,KE,3,263.2368
3,SHQ022,Left,KF,1,106.9800
4,SHQ022,Left,KF,2,91.9228
5,SHQ022,Left,KF,3,95.3516
6,SHQ022,Right,KE,1,293.0436
7,SHQ022,Right,KE,2,266.8235
8,SHQ022,Right,KE,3,258.9814
9,SHQ022,Right,KF,1,133.0231
